In [1]:
import psi4
import pandas as pd
import numpy as np
from lps_rscf import lps_solver

In [56]:
psi4.core.set_output_file('output.dat', False)

ATOMS = {
    'H':  {'mult': 2}, 
    'He': {'mult': 1},
    'Li': {'mult': 2}, 
    'Be': {'mult': 1},
    'B':  {'mult': 2}, 
    'C':  {'mult': 3},
    'N':  {'mult': 4}, 
    'O':  {'mult': 3},
    'F':  {'mult': 2}, 
    'Ne': {'mult': 1}
}

METHOD = "TFD0.2W"
TP = ['LDA_K_TF', 1.0]
LAMBDA = 0.20
EXC = ['LDA_X', 1.0, 'LDA_C_VWN', 0.0]
FA = [False, 1.0]
DIIS = True
MAX_ITER = 2000
DAMPING = [0.9, 0.9, 0.001]
D_guess = None
verbose=False

psi4.set_options({'basis': 'Chan2001', 
                  'DFT_SPHERICAL_POINTS': 6, 
                  'DFT_RADIAL_POINTS': 1000})

data_rows = []

for atom in ATOMS:
    
    if not df.empty:
        exists = df[
            (df['Atom'] == atom) & 
            (df['Method'] == METHOD) & 
            (df['Basis'] == psi4.core.get_global_option("BASIS"))
        ]
        if not exists.empty:
            print(f"Skipping {atom} (Already exists for {METHOD}/{psi4.core.get_global_option("BASIS")})")
            continue

    print(f"Calculating {atom} with {METHOD}...")
    MOL = psi4.geometry(f"0 {ATOMS[atom]['mult']}\n {atom}\nsymmetry c1")
    try:
        E, D, mu, iterations = lps_solver(MAX_ITER,TP,EXC,LAMBDA,MOL,DAMPING,FA,D_guess,DIIS,verbose)
        if iterations >= MAX_ITER:
            print("  !!! SCF failed to converge (Max cycles exceeded).")
        else:
            print(f"Calculated Energy: {E:.4f} Hartree")
            row = {
                "Method": METHOD,
                "Atom": atom,
                "Basis": psi4.core.get_global_option("BASIS"),
                "Grid_Sph": psi4.core.get_global_option("DFT_SPHERICAL_POINTS"),
                "Grid_Rad": psi4.core.get_global_option("DFT_RADIAL_POINTS"),
                "Energy,Ha": E,
                "ChemPot,Ha": mu,
                "Iterations": iterations,
                "DIIS": DIIS,
                "Damp_Start": DAMPING[0],
                "Damp_End": DAMPING[1],
                "Damp_Cutoff": DAMPING[2]
            }
            
            df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    
    except Exception as e:
        print(f"  !!! Failed {atom}. Error: {e}")
        continue

Calculating H with TFD0.2W...
Calculated Energy: -0.5666 Hartree
Calculating He with TFD0.2W...
Calculated Energy: -2.8183 Hartree
Calculating Li with TFD0.2W...
Calculated Energy: -7.3227 Hartree
Calculating Be with TFD0.2W...
Calculated Energy: -14.4841 Hartree
Calculating B with TFD0.2W...
Calculated Energy: -24.6284 Hartree
Calculating C with TFD0.2W...
Calculated Energy: -38.0332 Hartree
Calculating N with TFD0.2W...
Calculated Energy: -54.9428 Hartree
Calculating O with TFD0.2W...
Calculated Energy: -75.5765 Hartree
Calculating F with TFD0.2W...
Calculated Energy: -100.1345 Hartree
Calculating Ne with TFD0.2W...
Calculated Energy: -128.8014 Hartree


In [ ]:
display(df)

,Method,Atom,Basis,Grid_Sph,Grid_Rad,"Energy,Ha","ChemPot,Ha",Iterations,DIIS,Damp_Start,Damp_End,Damp_Cutoff
0,TFDW,H,CHAN2001,6,1000,-0.2618,-0.0714,14,True,0.1000,0.0000,0.0010
1,TFDW,He,CHAN2001,6,1000,-1.4774,-0.1082,14,True,0.1000,0.0000,0.0010
2,TFDW,Li,CHAN2001,6,1000,-4.1054,-0.1306,15,True,0.1000,0.0000,0.0010
3,TFDW,Be,CHAN2001,6,1000,-8.4922,-0.1453,18,True,0.1000,0.0000,0.0010
4,TFDW,B,CHAN2001,6,1000,-14.9258,-0.1556,51,True,0.1000,0.0000,0.0010
5,TFDW,C,CHAN2001,6,1000,-23.6568,-0.1633,44,True,0.1000,0.0000,0.0010
6,TFDW,O,CHAN2001,6,1000,-48.8831,-0.1737,133,True,0.9000,0.0000,0.0010
7,TFDW,N,CHAN2001,6,1000,-34.9084,-0.1691,104,True,0.9000,0.0000,0.0010
8,TFDW,F,CHAN2001,6,1000,-65.7674,-0.1775,601,True,0.1000,0.0000,0.0010
9,TFDW,Ne,CHAN2001,6,1000,-85.7343,-0.1807,126,True,0.9000,0.0000,0.0010


In [58]:
df.to_csv("chan2001_table1.csv", index=False)

In [ ]:
titles = list(df.index)
titles[6], titles[8] = titles[8], titles[6]
df = df.reindex(titles)
display(df)